NB.01: Wahlkampf mit Naive Bayes
	Training:
		Hypothesen H = {O, M}

		A-priori-Wahrscheinlichkeiten:
			P(O) = 4/7
			P(M) = 3/7

		Merkmal Alter:
			P(>=35 | O)	= 1/2
			P(<35 | O)	= 1/2
			P(>=35 | M) = 2/3
			P(<35 |	M)	= 1/3
		Merkmal Einkommen:
			P(hoch | O)		= 3/4
			P(niedrig | O)	= 1/4
			P(hoch | M)		= 1/3
			P(niedrig | M)	= 2/3
		Merkmal Bildung:
			P(Abitur | O)	= 1/4
			P(Bachelor | O)	= 1/4
			P(Master | O)	= 2/4
			P(Abitur | M)	= 2/3
			P(Bachelor | M)	= 1/3
			P(Master | M)	= 0/3

	Klassifikiation: <35, niedrig, Bachelor
		h = O : P(O) * P(<35 | O) * P(niedrig | O) * P(Bachelor | O) = 4/7 * 1/2 * 1/4 * 1/4 = 1/56 = 0.0178
		h = M : P(M) * P(<35 | M) * P(niedrig | M) * P(Bachelor | M) = 3/7 * 1/3 * 2/3 * 1/3 = 2/63 = 0.0317

		Der Klassifikator berechnet für jede Klasse die Wahrscheinlichkeit, dass sie unter den gegebenen Daten eintritt und wählt die Klasse mit der höchsten Wahrscheinlichkeit aus. Hier ist das Kandidat M.

In [ ]:
import pandas as pd
import numpy as np
import re

## Using Bernoulli NB, so only checkin if a word occurs in a text, not how many times
## Processing Dataset

df = pd.read_csv("spam_ham_dataset.csv")
texts = df['text']
labels = df['label']

# Words that don't convey much information
stopwords = {
    "the", "and", "is", "to", "of", "in", "a", "for", "on"
}

def clean_text(text):
    text = text.lower()     # To lowercase for consistency
    text = re.sub(r'[^\w\s]', '', text) # remove numbers, punctuation
    text = text.split()   # Split into individual words on spaces
    text = [t for t in text if t not in stopwords]    # Ignore stopwords

    return text # returns a List[str]

processed_texts = [set(clean_text(t)) for t in texts]

# Build a bag of words using the processed texts
word_bag = set()
for text in processed_texts:
    word_bag.update(text)
word_bag = sorted(word_bag)
print(f"Vocabulary: {len(word_bag)} Words")

# Build the feature matrix of word x text
word_to_index = {word: i for i, word in enumerate(word_bag)}    
feature_matrix = np.zeros((len(processed_texts), len(word_bag)), dtype=int)
for i, text in enumerate(processed_texts):
    for word in text:
        if word in word_to_index:
            feature_matrix[i, word_to_index[word]] = 1

feature_matrix_df = pd.DataFrame(feature_matrix, columns=word_bag)
feature_matrix_df['label'] = labels

print(feature_matrix_df.head())

## Classificator "training"

# count occurence of words for ham and spam
y = np.array(df['label'])
word_counts_ham  = feature_matrix[y == 'ham'].sum(axis=0)
word_counts_spam = feature_matrix[y == 'spam'].sum(axis=0)

# count total ham and spam entries
total_ham  = np.sum(y == 'ham')
total_spam = np.sum(y == 'spam')

# calculate a-priori probabilites : P(A)
P_ham = total_ham / len(y)
P_spam = total_spam / len(y)

# calculate probabilities for each word : P(B)
P_word = feature_matrix.mean(axis=0)

# calculate likelihoods for each word, given spam or ham (using laplace smoothing with P_word as p_i ) : P(B | A)
alpha = 1   # chosen arbitrarily
P_word_if_ham  = (word_counts_ham + alpha ) / (total_ham + 2 * alpha)
P_word_if_spam = (word_counts_spam + alpha) / (total_spam + 2 * alpha)

## Classificator prediction (posterior probability)

## TODO


Vocabulary: 50514 Words
   0  00  000  0000  000000  000000000002858  000000000049773  000080  000099  \
0  0   1    0     0       0                0                0       0       0   
1  0   0    0     0       0                0                0       0       0   
2  0   0    0     0       0                0                0       0       0   
3  0   0    0     0       0                0                0       0       0   
4  0   0    0     0       0                0                0       0       0   

   0001  ...  zynve  zyqtaqlt  zyrtec  zyyqywp  zzezrjok  zzn  zzo  zzocb  \
0     0  ...      0         0       0        0         0    0    0      0   
1     0  ...      0         0       0        0         0    0    0      0   
2     0  ...      0         0       0        0         0    0    0      0   
3     0  ...      0         0       0        0         0    0    0      0   
4     0  ...      0         0       0        0         0    0    0      0   

   zzso  zzsyt  
0     0  